# 00 · Dados e dobras (protocolo único)

**Sessão P1 · roda em CPU** (Colab → *Alterar tipo de ambiente* → CPU). Não gasta GPU.

O que este notebook faz, **uma única vez**:

1. clona o FDCPUnBGunshotDB no commit fixado `f679389` (e o leguwoi no `83d82f1`), **dentro da sessão** — nenhuma imagem vai para o Drive (a base não tem licença de redistribuição; cite Lira et al., 2025, IJLM);
2. lista as imagens como nos notebooks originais (rótulos pelas pastas), **ordenadas por `rel_path`**;
3. cruza com o CSV do dataset e anota as **4 anomalias** na coluna `obs`;
4. gera as dobras: `StratifiedGroupKFold(5, shuffle=True, random_state=42)`, `y = strat4` (0 saída, 1 encostado, 2 curta, 3 distância), `groups = case_id`, com a **regra pré-registrada** (cada dobra de teste com ≥ 2 casos de encostado e ≥ 1 imagem de cada classe; se a semente 42 falhar, a 1ª ≥ 43 que passe);
5. confere tudo (vazamento de casos, ±3 pp de saídas por dobra, contagens) e o *cache* 224×224;
6. grava `dados/folds_entrada_saida.csv` e `dados/dobras_meta.json` no Drive — **nunca sobrescreve** um arquivo diferente.

Colunas do CSV: `rel_path, case_id, y_es, y_dist (−1 nas saídas), strat4, fold, obs`. Ele é compartilhado pelos Prompts 1, 2 e 3.

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"  # pasta tcc-final (conta padrão, authuser=0)
RAIZ_TRABALHO = "/content"   # onde o dataset é clonado e o cache é gerado (disco local, temporário)
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"

In [ ]:
# Dependências fixadas (rode ANTES de importar qualquer coisa da albumentations)
!pip install -q "albumentations==2.0.8" "git+https://github.com/Blealtan/efficient-kan@7b6ce1c87f18c8bc90c208f6b494042344216b11"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, (
    "O protocolo_comum.py do Drive não é o publicado. Não rode nada; avise o P1.")
print("Dependências:", pc.checar_dependencias())
print("Protocolo", pc.PROTOCOLO_VERSAO, "| SHA-256", pc.sha256_protocolo()[:16], "…")

## 1. Clonar o dataset e o leguwoi (commits fixados)

In [ ]:
raiz_ds = f"{RAIZ_TRABALHO}/FDCPUnBGunshotDB"
print("FDCPUnBGunshotDB @", pc.clonar_fixado(pc.DATASET_URL, pc.DATASET_COMMIT, raiz_ds))
print("leguwoi          @", pc.clonar_fixado(pc.LEGUWOI_URL, pc.LEGUWOI_COMMIT, f"{RAIZ_TRABALHO}/leguwoi"))

## 2. Listagem e rótulos (pelas pastas, como nos notebooks originais)

In [ ]:
df = pc.listar_imagens(raiz_ds)
print(len(df), "imagens em", df.case_id.nunique(), "casos (ano.caso)")
print("y_es  :", df.y_es.value_counts().sort_index().to_dict())
print("y_dist:", df.y_dist.value_counts().sort_index().to_dict())
assert len(df) == 2554 and df.case_id.nunique() == 486, "contagem inesperada — o clone mudou?"
assert df.y_dist.value_counts().to_dict() == {-1: 671, 0: 42, 1: 232, 2: 1609}

## 3. Anomalias do CSV do dataset (coluna `obs`)

In [ ]:
anom = pc.anomalias_csv(raiz_ds, df)
for k, v in anom.items():
    print(f"- {k}: {v}")
df["obs"] = df["rel_path"].map(anom).fillna("")
if len(anom) != 4 or (df["obs"] != "").sum() != 4:
    raise RuntimeError(f"Esperava 4 anomalias e achei {len(anom)}. Nada foi gravado; avise o P1.")

## 4. Dobras (regra pré-registrada) e checagens

In [ ]:
dobras, info = pc.gerar_dobras(df)
print("Semente usada:", info["semente_usada"], "| tentativas:", [t["semente"] for t in info["tentativas"]])
chk = pc.checar_dobras(dobras)
display(pd.DataFrame(chk["por_dobra"]))
print("Fração global de saídas:", chk["frac_saida_global"])
print("Casos em mais de uma dobra:", chk["casos_em_mais_de_uma_dobra"])
col = pc.colisoes_chave_leguwoi(dobras)
print("Chave do leguwoi (só o número):", col)
if chk["avisos"]:
    raise RuntimeError("Checagem das dobras falhou (nada foi gravado): " + "; ".join(chk["avisos"]))

## 5. Cache 224×224 (só nesta sessão; nunca no Drive)

In [ ]:
t0 = time.time()
cache = pc.construir_cache(raiz_ds, list(dobras["rel_path"]))
assert pc.conferir_cache(cache, raiz_ds, list(dobras["rel_path"]), n=5), "cache ≠ A.Resize on-the-fly"
sha_cache = pc.sha256_array(cache)
print(f"Cache {cache.shape} em {time.time() - t0:.0f}s | SHA-256 {sha_cache[:16]}…")
avisos = [a for a in pc.checar_avisos_transforms(cache[:5]) if "utcnow" not in a]
print("Avisos inesperados da Albumentations:", avisos or "nenhum")
assert not avisos

## 6. KFold antigo nesta máquina (informativo)

O notebook original listava as pastas sem ordenar, e essa ordem muda entre máquinas do Colab. Os tamanhos de teste das 5 dobras antigas, deduzidos do log, são **496 / 636 / 459 / 465 / 498**. A ponte (notebook 01) só usa a dobra antiga se essa impressão digital conferir.

In [ ]:
dig = pc.conferir_impressao_digital(pc.dobras_antigas(raiz_ds))
print(dig)

## 7. Gravar no Drive (uma única vez)

In [ ]:
dir_dados = Path(DIR_DRIVE) / "dados"
dir_dados.mkdir(parents=True, exist_ok=True)
arq = dir_dados / pc.ARQUIVO_DOBRAS
colunas = ["rel_path", "case_id", "y_es", "y_dist", "strat4", "fold", "obs"]
novo = dobras[colunas].to_csv(index=False, lineterminator="\n")
if arq.exists():
    if arq.read_text(encoding="utf-8") != novo:
        raise RuntimeError("Já existe um folds_entrada_saida.csv DIFERENTE no Drive. Não sobrescrevo; avise o P1.")
    print("O arquivo de dobras já existe e é idêntico — nada a fazer.")
else:
    tmp = arq.with_name(arq.name + ".tmp")
    tmp.write_text(novo, encoding="utf-8")
    os.replace(tmp, arq)
    print("Gravado:", arq)
sha_dobras = pc.sha256_arquivo(arq)
arq_meta = dir_dados / pc.ARQUIVO_META_DOBRAS
if not arq_meta.exists():
    pc.gravar_json_atomico({
        "gerado_em": pc.agora_iso(),
        "dobras_sha256": sha_dobras,
        "semente_usada": info["semente_usada"],
        "tentativas": info["tentativas"],
        "regra": "StratifiedGroupKFold(5, shuffle=True) com y=strat4, groups=case_id; >=2 casos de encostado e >=1 imagem por classe em cada dobra de teste",
        "checagem": chk,
        "anomalias": anom,
        "colisoes_chave_leguwoi": col,
        "cache_sha256": sha_cache,
        "impressao_digital_antiga_nesta_maquina": dig,
        "dataset_commit": pc.DATASET_COMMIT,
        "leguwoi_commit": pc.LEGUWOI_COMMIT,
        "ambiente": pc.info_ambiente(),
    }, arq_meta)
    print("Gravado:", arq_meta)
else:
    print("dobras_meta.json já existe — mantido.")
print("SHA-256 do CSV de dobras:", sha_dobras)

## Próximo passo

Abra o **01_calibracao_L4** num ambiente **L4** e rode tudo. Ele faz a ponte fp32 × AMP (que decide a precisão de todas as rodadas) e mede o custo em unidades de computação.